# Vergleich von Aktien
## 1. Datenbeschaffung
### 1.1 Aktien und Zeitraum auswählen

In [1]:
import yfinance

tickers = {
    "AAPL": "Apple (USA)",
    "NVDA": "NVIDIA (USA)",
    "ASML": "ASML (Niederlande)",
    "SAP": "SAP (Deutschland)",
    "TM": "Toyota (Japan)",
    "SONY": "Sony (Japan)",
}

start_date = "2021-08-01"
end_date = "2026-07-31"

Wir betrachten sechs Aktien aus vier Ländern (USA, Niederlande, Deutschland, Japan) über einen Zeitraum von fünf Jahren (01.08.2021-31.07.2026). Die Auswahl deckt unterschiedliche Branchen ab (Technologie, Halbleiter, Software, Automobil), um im späteren Vergleich sowohl ähnliche als auch unterschiedliche Wertentwicklungen zeigen zu können.

### 1.2 Kursdaten herunterladen

In [2]:
data = yfinance.download(list(tickers.keys()), start=start_date, end=end_date, auto_adjust=True)
data.head()

[*********************100%***********************]  6 of 6 completed


Price            Close                                                \
Ticker            AAPL        ASML       NVDA         SAP       SONY   
Date                                                                   
2021-08-02  141.845886  734.308716  19.681925  131.754990  20.368086   
2021-08-03  143.639465  743.734741  19.746702  133.795822  20.227613   
2021-08-04  143.239822  758.952454  20.204113  134.577240  19.950579   
2021-08-05  143.346985  758.256104  20.565865  136.921463  20.063736   
2021-08-06  142.663635  747.312622  20.295797  134.558868  20.020815   

Price                         High                                     ...  \
Ticker              TM        AAPL        ASML       NVDA         SAP  ...   
Date                                                                   ...   
2021-08-02  162.836517  143.239774  740.214482  19.892198  132.940889  ...   
2021-08-03  166.480377  144.302290  744.612479  20.152299  133.814212  ...   
2021-08-04  163.105087  144.058611  761.900590  20.247962  135.183985  ...   
2021-08-05  163.713928  144.107290  768.646063  20.661534  137.252414  ...   
2021-08-06  161.797974  143.610562  751.043053  20.499095  135.634455  ...   

Price            Open                                       Volume          \
Ticker           NVDA         SAP       SONY          TM      AAPL    ASML   
Date                                                                         
2021-08-02  19.632098  131.506773  20.442223  162.943949  62880000  499900   
2021-08-03  19.671960  132.848938  20.405151  164.304813  64786600  533700   
2021-08-04  19.921092  134.108399  20.005206  163.570645  56368300  629700   
2021-08-05  20.429338  136.544544  20.284195  163.409531  46397700  723700   
2021-08-06  20.453254  135.101270  20.188597  162.173996  54126800  556900   

Price                                            
Ticker           NVDA      SAP     SONY      TM  
Date                                             
2021-08-02  217444000   382800  2037000  167500  
2021-08-03  301811000   742400  3666000  246900  
2021-08-04  231309000  1003100  3098500  194600  
2021-08-05  211435000   574200  2101500  128800  
2021-08-06  178497000   490300  2066000  183400  

[5 rows x 30 columns]

Die Kursdaten werden über die Python-Bibliothek `yfinance` abgerufen, die auf öffentlich verfügbare Daten von Yahoo Finance zugreift. Der Parameter `auto_adjust=True` sorgt dafür, dass die Kurse um Aktiensplits und Dividenden bereinigt sind. Ohne diese Bereinigung würden z. B. Aktiensplits (wie bei NVIDIA 2024) als künstliche Kurseinbrüche erscheinen.

### 1.3 Sanity-Check

In [3]:
print("Zeitraum von", data.index.min().date(), "bis", data.index.max().date())
print("Anzahl der Handelstage:", len(data))
print("Fehlende Werte je Aktie:")
print(data.isna().sum())

Zeitraum von 2021-08-02 bis 2026-07-30
Anzahl der Handelstage: 1254
Fehlende Werte je Aktie:
Price   Ticker
Close   AAPL      0
        ASML      0
        NVDA      0
        SAP       0
        SONY      0
        TM        0
High    AAPL      0
        ASML      0
        NVDA      0
        SAP       0
        SONY      0
        TM        0
Low     AAPL      0
        ASML      0
        NVDA      0
        SAP       0
        SONY      0
        TM        0
Open    AAPL      0
        ASML      0
        NVDA      0
        SAP       0
        SONY      0
        TM        0
Volume  AAPL      0
        ASML      0
        NVDA      0
        SAP       0
        SONY      0
        TM        0
dtype: int64


Vor der Weiterverarbeitung wird geprüft, ob die Daten vollständig sind und der erwartete Zeitraum abgedeckt ist.

## 2. Datenaufbereitung
### 2.1 Schlusskurse extrahieren und umbenennen

In [4]:
close = data["Close"].rename(columns = tickers)
close.head()

Ticker,Apple (USA),ASML (Niederlande),NVIDIA (USA),SAP (Deutschland),Sony (Japan),Toyota (Japan)
Date,,,,,,
2021-08-02,141.845886,734.308716,19.681925,131.754990,20.368086,162.836517
2021-08-03,143.639465,743.734741,19.746702,133.795822,20.227613,166.480377
2021-08-04,143.239822,758.952454,20.204113,134.577240,19.950579,163.105087
2021-08-05,143.346985,758.256104,20.565865,136.921463,20.063736,163.713928
2021-08-06,142.663635,747.312622,20.295797,134.558868,20.020815,161.797974


Aus den vollständigen Kursdaten wird nur der Schlusskurs weiterverwendet, da dieser für die Rendite- und Risikoberechnung die relevante Größe ist. Die Spalten werden umbenannt, um das Notebook lesbarer zu machen.

### 2.2 Tägliche Rendite

In [5]:
returns = close.pct_change().dropna()
returns.head()

Ticker,Apple (USA),ASML (Niederlande),NVIDIA (USA),SAP (Deutschland),Sony (Japan),Toyota (Japan)
Date,,,,,,
2021-08-03,0.012645,0.012837,0.003291,0.015490,-0.006897,0.022377
2021-08-04,-0.002782,0.020461,0.023164,0.005840,-0.013696,-0.020274
2021-08-05,0.000748,-0.000918,0.017905,0.017419,0.005672,0.003733
2021-08-06,-0.004767,-0.014432,-0.013132,-0.017255,-0.002139,-0.011703
2021-08-09,-0.000342,0.006894,-0.003486,0.001298,0.000097,-0.000830


Die tägliche Rendite zeigt, um wie viel Prozent sich der Kurs von einem Tag auf den nächsten verändert hat.

Beispiel:  
Der Apple-Kurs lag am 02.08.2021 bei 141,85 USD und am 03.08.2021 bei 143,64 USD.
Das entspricht einem Anstieg von 1,79 USD, also rund +1,26 %.

## 3. Kennzahlen
### 3.1 Kumulierte Rendite

In [6]:
cumulative_return = (1 + returns).cumprod() - 1
cumulative_return.tail()

Ticker,Apple (USA),ASML (Niederlande),NVIDIA (USA),SAP (Deutschland),Sony (Japan),Toyota (Japan)
Date,,,,,,
2026-07-24,1.347759,1.389758,9.509135,0.214375,0.030043,0.089559
2026-07-27,1.375183,1.251263,8.984287,0.297939,0.094359,0.118607
2026-07-28,1.397532,1.155701,9.009691,0.359873,0.114980,0.144092
2026-07-29,1.384207,1.111768,8.654035,0.411635,0.142474,0.184255
2026-07-30,1.350650,1.248972,8.909600,0.372851,0.117925,0.175719


Die kumulierte Rendite zeigt, was aus einem angelegten Betrag über die Zeit geworden wäre, wenn man alle täglichen Veränderungen mitnimmt.

Beispiel:  
Bei NVIDIA steht am Ende des Zeitraums eine kumulierte Rendite von rund 8,91 (also +891 %).
Das heißt: Aus 100 € investiert am 01.08.2021 wären bis zum 30.07.2026 etwa 991 € geworden.

Wichtig dabei:  
Die täglichen Veränderungen bauen aufeinander auf.
Ein Gewinn von einem Tag wirkt sich am nächsten Tag mit aus, weil er schon Teil des neuen Kurses ist.

### 3.2 Jährliche Volatilität

In [7]:
volatility = returns.std() * (252 ** 0.5)
volatility

Ticker
Apple (USA)           0.278409
ASML (Niederlande)    0.432816
NVIDIA (USA)          0.518987
SAP (Deutschland)     0.297359
Sony (Japan)          0.292453
Toyota (Japan)        0.271929
dtype: float64

Die Volatilität zeigt, wie stark der Kurs einer Aktie schwankt.

Beispiel:  
Toyota hat mit rund 27 % die niedrigste Volatilität, der Kurs bewegt sich vergleichsweise gleichmäßig.
NVIDIA hat mit rund 52 % fast doppelt so hohe Schwankungen, der Kurs macht größere Sprünge nach oben und unten.

Hohe Volatilität bedeutet höheres Risiko, aber oft auch die Chance auf höhere Gewinne.
Bei NVIDIA sieht man beides: die höchste Schwankung und die höchste Rendite.

### 3.3 Sharpe Ratio

In [8]:
annualized_return = returns.mean() * 252
sharpe_ratio = annualized_return / volatility
sharpe_ratio

Ticker
Apple (USA)           0.756079
ASML (Niederlande)    0.593050
NVIDIA (USA)          1.146121
SAP (Deutschland)     0.363118
Sony (Japan)          0.222402
Toyota (Japan)        0.254728
dtype: float64

Die Sharpe Ratio zeigt, wie viel Rendite eine Aktie pro Einheit eingegangenem Risiko liefert (hier vereinfacht mit risikofreiem Zins = 0).

Beispiel:  
NVIDIA hat mit 1,15 die höchste Sharpe Ratio, trotz der hohen Volatilität wird das Risiko durch die außergewöhnlich hohe Rendite mehr als ausgeglichen. Apple liegt mit 0,76 an zweiter Stelle, obwohl die Volatilität niedriger ist als bei ASML (0,59) oder SAP (0,36). Apple liefert also nach NVIDIA die "effizienteste" Rendite pro Risiko.

Sony (0,22) und Toyota (0,25) haben die niedrigsten Sharpe Ratios. Eher geringe Renditen bei einer Volatilität, die nicht wesentlich niedriger ist als bei den anderen Aktien. Das eingegangene Risiko lohnt sich hier vergleichsweise am wenigsten.

### 3.4 Maximaler Drawdown

In [9]:
running_max = close.cummax()
drawdown = (close - running_max) / running_max
max_drawdown = drawdown.min()
max_drawdown

Ticker
Apple (USA)          -0.333605
ASML (Niederlande)   -0.568618
NVIDIA (USA)         -0.663351
SAP (Deutschland)    -0.522570
Sony (Japan)         -0.505593
Toyota (Japan)       -0.368022
dtype: float64

Der maximale Drawdown zeigt den größten Wertverlust, den eine Aktie vom bisherigen Höchststand bis zu ihrem nachfolgenden Tiefpunkt erlitten hat.

Beispiel:  
NVIDIA hatte mit -66 % den stärksten Einbruch. Wer am absoluten Höchstpunkt gekauft hätte, hätte zwischenzeitlich zwei Drittel des Werts verloren, bevor sich die Aktie erholte.
Apple war mit -33 % am stabilsten.

Auffällig:  
SAP (-52 %) und Sony (-51 %) hatten trotz vergleichsweise niedriger Volatilität und niedriger Sharpe Ratio fast genauso tiefe Einbrüche wie ASML (-57 %).

Das zeigt:  
Volatilität misst eher die täglichen Schwankungen, während der maximale Drawdown einen anhaltenden Abwärtstrend über einen längeren Zeitraum erfasst.
Beide Kennzahlen zusammen ergeben ein vollständigeres Risikobild als jede für sich allein.

### 3.5 Korrelationsmatrix

In [10]:
correlation_matrix = returns.corr()
correlation_matrix

Ticker,Apple (USA),ASML (Niederlande),NVIDIA (USA),SAP (Deutschland),Sony (Japan),Toyota (Japan)
Ticker,,,,,,
Apple (USA),1.000000,0.482137,0.485503,0.412071,0.435766,0.397315
ASML (Niederlande),0.482137,1.000000,0.659557,0.396505,0.393712,0.389986
NVIDIA (USA),0.485503,0.659557,1.000000,0.384321,0.392709,0.338143
SAP (Deutschland),0.412071,0.396505,0.384321,1.000000,0.402798,0.307407
Sony (Japan),0.435766,0.393712,0.392709,0.402798,1.000000,0.465840
Toyota (Japan),0.397315,0.389986,0.338143,0.307407,0.465840,1.000000


Die Korrelationsmatrix zeigt, wie stark sich die Renditen der sechs Aktien im Gleichlauf bewegen.

Auffällig:  
ASML und NVIDIA haben mit 0,66 die höchste Korrelation im gesamten Vergleich, deutlich höher als jedes andere Aktienpaar. Das bestätigt die eingangs aufgestellte Hypothese: Beide Unternehmen sind eng mit dem Halbleiterzyklus verbunden (ASML liefert die Maschinen, mit denen u. a. NVIDIA-Chips hergestellt werden), wodurch sich ihre Kursbewegungen stärker ähneln als bei branchenfremden Paaren.

Am schwächsten korreliert sind NVIDIA und Toyota (0,34) sowie SAP und Toyota (0,31). Zwei Unternehmen aus völlig unterschiedlichen Branchen und Weltregionen bewegen sich am unabhängigsten voneinander.

Insgesamt sind alle Korrelationen positiv, kein Aktienpaar bewegt sich gegenläufig. Das ist typisch für global gehandelte Aktien, die alle von übergeordneten Markteinflüssen (z. B. Zinsentscheidungen, Konjunktur) mitbestimmt werden. Der Unterschied liegt im Ausmaß, branchennahe Aktien bewegen sich deutlich synchroner als branchenfremde.